In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10

In [2]:
# Define Tranformers-> which tranforms to apply on images

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),
                        (0.5,0.5,0.5))
])

In [3]:
#datasets

trainset = CIFAR10(root="./data",train=True,download=True,transform=transform)
testset  = CIFAR10(root="./data",train=True,download=True,transform=transform)

100%|██████████| 170M/170M [00:02<00:00, 85.0MB/s]


In [6]:
# lets see the trainset

trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [7]:
 # dataloader

from torch.utils.data import DataLoader

train_loader = DataLoader(trainset,batch_size=64,shuffle=True)
test_loader = DataLoader(testset,batch_size=64)

# Build the model

In [8]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layer = nn.Sequential(

        #1 layer cannot capture complex patterns well therfore we added 3 layers
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    #image will of 4*4 and feature map will be of size 128 when we fllaten these we get 4*4*128 input features

    self.fc_layer = nn.Sequential(
        nn.Linear(128*4*4,256),
        nn.ReLU(),

        #output layer
        nn.Linear(256,10)
    )

  def forward(self,x):
    x = self.conv_layer(x)
    # now fllaten the matrix
    x = x.view(x.size(0),-1)

    x = self.fc_layer(x)
    return x


In [9]:
# create the obj of the CNN model
model = CNN()


In [10]:
# now select the loss function and optimize the model
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [11]:
epoch=10

for i in range(epoch):
  loss_per_epoch =0.0
  model.train()

  for image,labels in train_loader:
    # foward propagation
    output = model.forward(image)

    optimizer.zero_grad()

    loss = criterion(output,labels)
    loss_per_epoch += loss.item()

    loss.backward()
    optimizer.step() # update params

  val_loss_per_epoch=0.0

  model.eval()
  with torch.no_grad():
    for image,labels in test_loader:

      output = model.forward(image)
      loss = criterion(output,labels)
      val_loss_per_epoch += loss.item()

  print(f"loss of {i+1} is: {loss_per_epoch/len(train_loader)} and validation loss: {val_loss_per_epoch/len(test_loader)}")

loss of 1 is: 1.3650082528133831 and validation loss: 1.0105096472193822
loss of 2 is: 0.9335055400038619 and validation loss: 0.7844098327333665
loss of 3 is: 0.7474212815313388 and validation loss: 0.6030298242407381
loss of 4 is: 0.6196876521915426 and validation loss: 0.5436207645041559
loss of 5 is: 0.5115597798581928 and validation loss: 0.39378122392746495
loss of 6 is: 0.4094120475565991 and validation loss: 0.30829062680606645
loss of 7 is: 0.3335459110091257 and validation loss: 0.2593763351268933
loss of 8 is: 0.25417418748407106 and validation loss: 0.1874393369535656
loss of 9 is: 0.19514961169599115 and validation loss: 0.16567158494192316
loss of 10 is: 0.1569117295682011 and validation loss: 0.12831074658238217


### Best Model is -> the Model having lowest Validation loss